In [ ]:
import pandas as pd
import numpy as np

from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)
from scipy.special import softmax

import torch
from torch.utils.data import Dataset

import torch.nn as nn

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

SEED = 42
MAX_LEN = 128

In [ ]:
MODELS = [
    "allegro/herbert-base-cased",
    "dkleczek/bert-base-polish-cased-v1",
    "sdadas/polish-roberta-base-v2"
]

In [ ]:
df = pd.read_csv("hate_train.csv")

train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    stratify=df["label"]
)

## Ważenie przykładów

In [ ]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print(class_weights)

In [ ]:


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [ ]:

class HateDataset(Dataset):

    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=MAX_LEN
        )

        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return item

def compute_metrics(pred):

    labels = pred.label_ids
    preds = pred.predictions.argmax(axis=1)

    acc = accuracy_score(labels, preds)
    precision = precision_score(labels, preds, zero_division=0)
    recall = recall_score(labels, preds, zero_division=0)
    f1 = f1_score(labels, preds, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(labels, preds).ravel()

    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    probs = softmax(pred.predictions, axis=1)[:, 1]
    preds = probs >= 0.5

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probs),
        "pr_auc": average_precision_score(labels, probs)
    }

## Porównanie modeli

In [ ]:
results = []

for model_name in MODELS:

    print("=" * 50)
    print(model_name)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

    val_dataset = HateDataset(
        val_df["sentence"],
        val_df["label"].values,
        tokenizer
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    args = TrainingArguments(
        output_dir=f"./tmp_{model_name.split('/')[-1]}",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        report_to="none"
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()

    metrics = trainer.evaluate()

    results.append({
        "model": model_name,
        "f1": metrics["eval_f1"],
        "accuracy": metrics["eval_accuracy"]
    })

In [ ]:

results_df = pd.DataFrame(results)

print("\nRESULTS")
print(results_df.sort_values("f1", ascending=False))

best_model_name = results_df.sort_values(
    "f1",
    ascending=False
).iloc[0]["model"]

print(f"\nBEST MODEL: {best_model_name}")

## Analiza najlepszego modelu

In [ ]:
print(best_model_name)

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

train_dataset = HateDataset(
        train_df["sentence"],
        train_df["label"].values,
        tokenizer
    )

val_dataset = HateDataset(
    val_df["sentence"],
    val_df["label"].values,
    tokenizer
)

model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir=f"./best_{best_model_name.split('/')[-1]}",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=20,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

metrics = trainer.evaluate()

print(metrics)

In [ ]:
from sklearn.metrics import classification_report

predictions = trainer.predict(val_dataset)

y_true = predictions.label_ids
y_pred = predictions.predictions.argmax(axis=1)

print(classification_report(y_true, y_pred))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Predicted 0", "Predicted 1"]
)

print(cm_df)

## Najlepszy model

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(best_model_name)

full_dataset = HateDataset(
    df["sentence"],
    df["label"].values,
    tokenizer
)

best_model = AutoModelForSequenceClassification.from_pretrained(
    best_model_name,
    num_labels=2
)

args = TrainingArguments(
    output_dir="./final_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none"
)

trainer = Trainer(
    model=best_model,
    args=args,
    train_dataset=full_dataset
)

trainer.train()

## Predykcje

In [ ]:
with open("hate_test_data.txt", "r", encoding="utf8") as f:
    test_texts = [line.strip() for line in f]

enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=MAX_LEN,
    return_tensors="pt"
)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

best_model.to(device)

enc = {
    k: v.to(device)
    for k, v in enc.items()
}

with torch.no_grad():
    outputs = best_model(**enc)

preds = outputs.logits.argmax(dim=1).cpu().numpy()

pd.DataFrame(preds).to_csv(
    "pred.csv",
    header=False,
    index=False
)
